In [1]:
import os
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'

In [2]:
import sys
sys.path.append('../')

In [3]:
import MeshFEM
import mesh, mesh_energy, py_newton_optimizer, viewer, param_utils
import parametrization, benchmark
import numpy as np

In [4]:
import continuation_parametrization, flip_avoiding_step_length

In [5]:
from Benchmark import helper_funcs

# Read Mesh and initialization

In [6]:
mesh_path = '../../../Models/PPdata/data1/fig12_a.obj'

In [7]:
m = helper_funcs.read_mesh(mesh_path)

In [8]:
uv = mesh_energy.NodalVars(m, 2)

In [9]:
bdry_uv = helper_funcs.getBDdataOnNormalizedCircle(m)
uv_init = helper_funcs.tutteInitialization(m, bdry_uv)

In [10]:
uv.setVars(uv_init.ravel())

# Prob Set up

In [11]:
param = continuation_parametrization.symmetric_dirichlet_param(m, uv)
objectives = [param]

In [12]:
# Construct parametrization energy and problem
prob = py_newton_optimizer.NewtonMultiobjectiveProblem(uv, objectives)
opt = prob.optimizer()

In [13]:
prob.initialFeasibleStepLengthComputer = flip_avoiding_step_length.FlipAvoidingStepLength(m.elements())
prob.initialFeasibleStepLengthComputer.backoffFactor = 0.95

In [14]:
# prob.hessianShift = 1e-9
# prob.useRelativeHessianShift = True

param.elementHessianShift = 1e-8

In [15]:
prob.objective()

13574.777401090383

In [16]:
DEGREE = 0

In [17]:
# opt.options.hessianProjectionController.numConsecutiveIndefiniteStepsBeforeEnable = 0
# opt.options.hessianProjectionController.numProjectionStepsBeforeDisable = 2
# opt.options.hessianProjectionController.startWithProjectionActive = False

opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAlways()

In [18]:
# benchmark.reset()
# param.setInterpolatedReference(0, uv_init.ravel())
# opt.options.hessianProjectionController.reset()
# prob.invalidateCachedHessian()
# opt.update_factorizations()
# print(prob.hessianWasProjected)
# benchmark.report()

## read l from file

In [19]:
l_arr = np.loadtxt('cm_only_true_area/fig12_a_pp_t.txt')

In [20]:
stats_trace = []

In [21]:
benchmark.reset()

param.setInterpolatedReference(0, uv_init.ravel())
prob.setVars(uv_init.ravel())

for it, l in enumerate(l_arr):
    prob.invalidateCachedHessian()
    # opt.options.hessianProjectionController.reset()
    opt.update_factorizations()
    
    print('l: ', l)
    param.setInterpolatedReference(l, prob.getVars())
    
    x0 = prob.getVars()
    opt.options.niter = 1
    opt.options.gradTol = 1e-6
    opt.optimize()
    
    # eval prob at 1.0
    param.setInterpolatedReference(1.0, uv_init.ravel())
    stats_trace.append({
        'iter': it,
        'l': float(l),
        'energy': prob.energy(),
        'grad_norm': np.linalg.norm(prob.gradient()),
    })
    # print(f'energy: {prob.energy()}; gradient_norm: {np.linalg.norm(prob.gradient())}')

benchmark.report()


l:  0.0
0	2	5.44422e-10	0	0	1
l:  0.22502211
0	2.95348	10370.1	0.43598	0	1
1	2.58067	5148.68	0.43598	0	1
l:  0.23248411
0	2.638	6031.07	1	0	1
1	2.21189	1807.3	1	0	1
l:  0.25787977
0	2.3144	3046.82	0.833527	0	1
1	2.13997	1384.75	0.833527	0	1
l:  0.26479598
0	2.15797	1595.9	1	0	1
1	2.07135	654.56	1	0	1
l:  0.28109113
0	2.09479	914.791	1	0	1
1	2.0558	379.277	1	0	1
l:  0.29782727
0	2.07158	536.009	1	0	1
1	2.05188	223.452	1	0	1
l:  0.31196354
0	2.06118	299.881	1	0	1
1	2.05126	125.764	1	0	1
l:  0.32892431
0	2.06088	179.581	1	0	1
1	2.05294	75.3816	1	0	1
l:  0.35110231
0	2.0662	120.661	1	0	1
1	2.05595	50.5991	1	0	1
l:  0.36985978
0	2.06611	75.5837	1	0	1
1	2.05815	31.6625	1	0	1
l:  0.39286495
0	2.07127	52.0312	1	0	1
1	2.06084	21.7532	1	0	1
l:  0.43353328
0	2.09521	52.4905	1	0	1
1	2.06586	21.8356	1	0	1
l:  0.46736438
0	2.09491	45.3268	1	0	1
1	2.06795	18.8073	1	0	1
l:  0.50608477
0	2.1031	43.3257	1	0	1
1	2.06976	17.725	1	0	1
l:  0.55359666
0	2.12052	49.1235	1	0	1
1	2.07171	19.767	1	0	1
l:  0.6476

# Debug

In [22]:
for st in stats_trace:
    print(f"{st['iter']}    {st['l']}    {st['energy']}    {st['grad_norm']}")

0    0.0    13574.777401089264    743990172534.2422
1    0.22502211    8675.219739358472    223078962429.99844
2    0.23248411    3808.4893221726343    37592164495.57476
3    0.25787977    2226.707990088023    16689062863.577879
4    0.26479598    1243.6833933566377    6863808535.797181
5    0.28109113    718.9723458805187    2864440996.9974537
6    0.29782727    429.30248420208187    1200809311.3821337
7    0.31196354    268.9204751385802    506133469.50081617
8    0.32892431    176.72576802777533    213254404.59602326
9    0.35110231    119.71009069454108    89473608.60413224
10    0.36985978    85.23196347613516    37644238.90033149
11    0.39286495    61.83065372154712    15774045.791592052
12    0.43353328    42.25329615746629    6387568.352328702
13    0.46736438    29.329055292866745    2559402.2454597843
14    0.50608477    20.33565814809964    973732.7626870641
15    0.55359666    13.926828781634516    350275.36230934685
16    0.64763905    8.825088028670518    106780.78973380

In [23]:
# TODO:
# Try attenuating the Hessian projection amount (interpolate down to no projection)